# Strategy Research R2: Volatility-Scaled Momentum (Core)

| Field | Value |
|-------|-------|
| **Researcher** | Elena |
| **Strategy** | Volatility-Scaled Momentum (Core) |
| **Round** | 2 (PM Challenge Response) |
| **Date** | 2026-03-15 |
| **Folder** | `research/strategies/vol_scaled_momentum_2026-03-13_conditional/` |

## Round 2 Changes

| # | Fix | Detail |
|---|-----|--------|
| 1 | **Portfolio Vol Targeting** | `target_vol=0.12` scales exposure when realized vol > 12% |
| 2 | **Turnover from weight diffs** | `abs(w_t - w_{t-1}).sum() / 2` per rebalance |
| 3 | **Risk-free adjusted Sharpe** | rf = 5.3% annualized (3M T-bill) |
| 4 | **WF outlier analysis** | Window 6 (COVID recovery) excluded in robustness check |
| 5 | **Weight distribution** | Investigate binding max_weight constraints |
| 6 | **EW benchmark** | Equal-weight buy-and-hold of same 28 stocks |

## Hypothesis & Literature Review

### Economic Rationale

This strategy combines three academically robust equity factors:

1. **Inverse Volatility** (Ang et al. 2006, JF): Low-volatility stocks earn higher risk-adjusted returns. Weight = 0.40.
2. **Risk-Adjusted Momentum** (Jegadeesh & Titman 1993; Sharpe-scaled per Barroso & Santa-Clara 2015): 12-1 month returns divided by trailing volatility. Weight = 0.40.
3. **Mean-Reversion Dampener** (Jegadeesh 1990): Short-term (20-day) reversal penalizes extended stocks. Weight = -0.20.

### Key Addition in R2: Portfolio Vol Targeting

Moreira & Muir (2017, JF) show that **scaling portfolio exposure inversely to realized volatility** substantially improves Sharpe ratios. The mechanism: volatility is persistent but NOT compensated (no vol-risk premium in equities), so reducing exposure in high-vol states avoids uncompensated risk.

R1 performed vol-scaled *stock selection* (inverse-vol signal) but NOT vol-scaled *exposure management*. R2 adds `target_vol=0.12` which reduces gross exposure when portfolio realized vol exceeds 12% annualized. This directly addresses the worst-regime and max-drawdown failures from R1.

### Who Loses Money?

Momentum losers (stocks with negative 12-month returns) and high-volatility stocks. These tend to be:
- Speculative growth stocks with uncertain earnings
- Distressed companies with high leverage
- Stocks with negative analyst revisions

The strategy also reduces exposure during high-vol regimes, forgoing gains during V-shaped recoveries.

In [1]:
# Cell 3: Setup & Imports
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = '/Users/zelin/Desktop/PA Investment/Invest_strategy'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# === STRATEGY CONFIGURATION (pre-committed, DO NOT TUNE) ===
STRATEGY_NAME = 'vol_scaled_momentum_core'
TICKERS = [
    'AAPL','MSFT','GOOGL','AMZN','META','BRK-B','JPM','JNJ','UNH','XOM',
    'CVX','PG','V','MA','HD','BAC','WMT','KO','PEP','ABBV',
    'MRK','TMO','COST','AVGO','CSCO','TXN','ACN','LIN','NEE','MDT'
]
START_DATE = '2010-01-01'
END_DATE = '2024-12-31'
IS_END = '2021-12-31'
OOS_START = '2022-01-01'

# Pre-committed signal weights (from proposal.md -- DO NOT TUNE)
WEIGHT_INVVOL = 0.40
WEIGHT_MOMENTUM = 0.40
WEIGHT_MEANREV = -0.20

# Pre-committed optimization parameters
RISK_AVERSION = 3.0
MAX_WEIGHT = 0.12
MIN_WEIGHT = 0.0
REBALANCE_FREQ = '2M'  # Bi-monthly

# NEW R2: Vol targeting
TARGET_VOL = 0.12  # 12% annualized -- Moreira & Muir (2017)

# Cost assumptions
FIXED_COST_PER_TRADE = 0.005
PROP_COST_BPS = 10.0

# Risk-free rate (Challenge #3)
RISK_FREE_RATE = 0.053  # 5.3% annualized, 3M T-bill

N_TRIALS = 1

print(f'Strategy: {STRATEGY_NAME}')
print(f'Universe: {len(TICKERS)} tickers')
print(f'Period: {START_DATE} to {END_DATE}')
print(f'IS/OOS split: {IS_END} | {OOS_START}')
print(f'Signal weights: InvVol={WEIGHT_INVVOL}, Mom={WEIGHT_MOMENTUM}, MeanRev={WEIGHT_MEANREV}')
print(f'Risk-free rate: {RISK_FREE_RATE:.1%}')
print(f'Target vol: {TARGET_VOL:.0%}')
print(f'Rebalance: {REBALANCE_FREQ} (bi-monthly)')

Strategy: vol_scaled_momentum_core
Universe: 30 tickers
Period: 2010-01-01 to 2024-12-31
IS/OOS split: 2021-12-31 | 2022-01-01
Signal weights: InvVol=0.4, Mom=0.4, MeanRev=-0.2
Risk-free rate: 5.3%
Target vol: 12%
Rebalance: 2M (bi-monthly)


In [2]:
# Cell 4: Data Loading & Inspection
import yfinance as yf

print('Downloading price data from yfinance...')
raw = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True, progress=True)

if isinstance(raw.columns, pd.MultiIndex):
    prices = raw['Close']
else:
    prices = raw[['Close']]

if isinstance(prices.columns, pd.MultiIndex):
    prices.columns = prices.columns.get_level_values(-1)

prices = prices.ffill(limit=5)
missing_pct = prices.isnull().mean()
good_tickers = missing_pct[missing_pct < 0.05].index.tolist()
prices = prices[good_tickers].dropna()

n_assets = len(prices.columns)
print(f'\nShape: {prices.shape}')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')
print(f'Trading days: {len(prices)}')
print(f'Tickers retained: {n_assets} / {len(TICKERS)}')
dropped = set(TICKERS) - set(prices.columns)
if dropped:
    print(f'Dropped: {dropped}')

[                       0%                       ]

[                       0%                       ]

[*****                 10%                       ]  3 of 30 completed

[********              17%                       ]  5 of 30 completed

[***********           23%                       ]  7 of 30 completed

...

[**********************90%******************     ]  27 of 30 completed

[**********************93%********************   ]  28 of 30 completed

[**********************97%********************** ]  29 of 30 completed

[*********************100%***********************]  30 of 30 completed


Shape: (3773, 28)
Date range: 2010-01-04 to 2024-12-30
Trading days: 3773
Tickers retained: 28 / 30
Dropped: {'ABBV', 'META'}


In [3]:
# Cell 5: Signal Construction
from backtests.strategies.signals import VolatilitySignal, MomentumSignal, MeanReversionSignal

vol_signal = VolatilitySignal(lookback=60)
raw_vol = vol_signal.compute(prices)

mom_signal = MomentumSignal(lookback=252, skip=21)
raw_mom = mom_signal.compute(prices)
trailing_vol = prices.pct_change().rolling(252, min_periods=252).std() * np.sqrt(252)
raw_sharpe_mom = raw_mom / trailing_vol.replace(0, np.nan)
raw_sharpe_mom = raw_sharpe_mom.replace([np.inf, -np.inf], np.nan)

mr_signal = MeanReversionSignal(lookback=20)
raw_mr = mr_signal.compute(prices)

MIN_PERIODS_ZSCORE = 60

def expanding_zscore(signal_df, min_periods=MIN_PERIODS_ZSCORE):
    """Two-stage normalization: cross-sectional z-score then expanding time-series."""
    cs_mean = signal_df.mean(axis=1)
    cs_std = signal_df.std(axis=1)
    cs_zscore = signal_df.sub(cs_mean, axis=0).div(cs_std.replace(0, np.nan), axis=0)
    exp_mean = cs_zscore.expanding(min_periods=min_periods).mean()
    exp_std = cs_zscore.expanding(min_periods=min_periods).std()
    return (cs_zscore - exp_mean) / exp_std.replace(0, np.nan)

z_vol = expanding_zscore(raw_vol)
z_mom = expanding_zscore(raw_sharpe_mom)
z_mr = expanding_zscore(raw_mr)

WARMUP = 252 + MIN_PERIODS_ZSCORE
print(f'Warmup: {WARMUP} days | First valid: {prices.index[WARMUP].date()}')
print(f'Signal shapes: vol={raw_vol.shape}, mom={raw_sharpe_mom.shape}, mr={raw_mr.shape}')

Warmup: 312 days | First valid: 2011-03-30
Signal shapes: vol=(3773, 28), mom=(3773, 28), mr=(3773, 28)


In [4]:
# Cell 6: Alpha Blending with Pre-Committed Weights
alpha = (WEIGHT_INVVOL * z_vol +
         WEIGHT_MOMENTUM * z_mom +
         WEIGHT_MEANREV * z_mr)
alpha = alpha.iloc[WARMUP:]

print(f'Alpha shape: {alpha.shape}')
print(f'Alpha range: {alpha.index[0].date()} to {alpha.index[-1].date()}')
print(f'Pre-committed weights: InvVol={WEIGHT_INVVOL}, Mom={WEIGHT_MOMENTUM}, MeanRev={WEIGHT_MEANREV}')
print('IMPORTANT: These weights were set BEFORE seeing any backtest results.')

Alpha shape: (3461, 28)
Alpha range: 2011-03-30 to 2024-12-30
Pre-committed weights: InvVol=0.4, Mom=0.4, MeanRev=-0.2
IMPORTANT: These weights were set BEFORE seeing any backtest results.


In [5]:
# Cell 7: Backtest with Vol Targeting (core R2 fix)
from backtests.builder import PortfolioBuilder, PortfolioConfig
from backtests.costs import CompositeCostModel, ProportionalCostModel, FixedCostModel

config = PortfolioConfig(
    universe=list(prices.columns),
    signals=[],
    optimization='mean_variance',
    risk_aversion=RISK_AVERSION,
    max_weight=MAX_WEIGHT,
    min_weight=MIN_WEIGHT,
    target_gross=1.0,
    rebalance_frequency=REBALANCE_FREQ,
    turnover_penalty=0.5,
    initial_cash=100000,
    commission=0.001,
)

builder = PortfolioBuilder(config=config)
builder.prices = prices
builder.signals = {'vol_scaled_momentum': alpha}
builder.weights = pd.Series(1.0 / n_assets, index=prices.columns)

cost_model = CompositeCostModel(
    models=(
        FixedCostModel(cost_per_trade=FIXED_COST_PER_TRADE),
        ProportionalCostModel(cost_bps=PROP_COST_BPS),
    )
)

backtest_start = str(prices.index[WARMUP].date())

# --- Run WITH vol targeting (R2) ---
print(f'Running backtest WITH target_vol={TARGET_VOL:.0%}')
print(f'Period: {backtest_start} to {END_DATE}')
print(f'Rebalance: {REBALANCE_FREQ} | Dynamic reoptimize: True')

result = builder.backtest(
    start_date=backtest_start,
    end_date=END_DATE,
    cost_model=cost_model,
    dynamic_reoptimize=True,
    target_vol=TARGET_VOL,
)

# --- Also run WITHOUT vol targeting for comparison ---
builder_noTV = PortfolioBuilder(config=config)
builder_noTV.prices = prices
builder_noTV.signals = {'vol_scaled_momentum': alpha}
builder_noTV.weights = pd.Series(1.0 / n_assets, index=prices.columns)

result_noTV = builder_noTV.backtest(
    start_date=backtest_start,
    end_date=END_DATE,
    cost_model=cost_model,
    dynamic_reoptimize=True,
    target_vol=None,  # no vol targeting
)

print(f'\nBacktest complete.')
print(f'  With target_vol:    {result["n_days"]} days')
print(f'  Without target_vol: {result_noTV["n_days"]} days')

Running backtest WITH target_vol=12%
Period: 2011-03-30 to 2024-12-31
Rebalance: 2M | Dynamic reoptimize: True



Backtest complete.
  With target_vol:    3460 days
  Without target_vol: 3460 days


In [6]:
# Cell 8: Weight Matrix & Proper Turnover (Challenge #2)
# Replicate weight loop to extract weight_matrix for turnover computation

daily_returns = result['daily_returns']
ret_df = prices.pct_change().loc[daily_returns.index]
common_assets = [a for a in builder.weights.index if a in ret_df.columns]
rebal_dates = ret_df.resample(REBALANCE_FREQ).last().dropna().index

# With vol targeting
weight_matrix = pd.DataFrame(0.0, index=ret_df.index, columns=common_assets)
curr_w = pd.Series(1.0 / len(common_assets), index=common_assets)
VOL_WINDOW = 60

for idx_num, date in enumerate(ret_df.index):
    is_rebal = date in rebal_dates or date == ret_df.index[0]
    if is_rebal:
        try:
            new_w = builder._optimize_weights_as_of(
                as_of_date=str(date.date()),
                common_assets=common_assets,
            )
            curr_w = new_w.reindex(common_assets).fillna(0)
        except Exception:
            pass
        # Apply vol targeting
        if idx_num >= VOL_WINDOW:
            trailing_rets = ret_df.iloc[idx_num - VOL_WINDOW:idx_num][common_assets]
            trailing_w = weight_matrix.iloc[idx_num - VOL_WINDOW:idx_num]
            port_rets = (trailing_w * trailing_rets).sum(axis=1)
            port_vol = port_rets.std() * np.sqrt(252)
            if port_vol > 0:
                scale = min(1.0, TARGET_VOL / port_vol)
                curr_w = curr_w * scale
    weight_matrix.loc[date] = curr_w

# Proper turnover: sum of |w_t - w_{t-1}| / 2 at each rebalance
weight_diffs = weight_matrix.diff().abs()
weight_diffs.iloc[0] = weight_matrix.iloc[0].abs()
# One-way turnover: sum of positive weight changes (= half of two-way)
daily_turnover_oneway = weight_diffs.sum(axis=1) / 2.0
n_days = len(daily_returns)
annual_turnover = daily_turnover_oneway.sum() / (n_days / 252) * 100

print(f'Proper annual turnover (one-way, weight-diff): {annual_turnover:.1f}%')
print(f'Rebalance dates used: {len([d for d in ret_df.index if d in rebal_dates])}')
print(f'Gross exposure range: [{weight_matrix.sum(axis=1).min():.3f}, {weight_matrix.sum(axis=1).max():.3f}]')
print(f'Avg gross exposure: {weight_matrix.sum(axis=1).mean():.3f}')

Proper annual turnover (one-way, weight-diff): 178.2%
Rebalance dates used: 59
Gross exposure range: [0.200, 1.000]
Avg gross exposure: 0.861


In [7]:
# Cell 9: Core Metrics — RF-Adjusted, IS/OOS Split
daily_rf = RISK_FREE_RATE / 252
excess_returns = daily_returns - daily_rf

# Full-sample
total_return = result['total_return']
ann_return = result['annualized_return']
ann_vol = daily_returns.std() * np.sqrt(252)
sharpe_raw = daily_returns.mean() / daily_returns.std() * np.sqrt(252) if daily_returns.std() > 0 else 0
sharpe_adj = excess_returns.mean() / excess_returns.std() * np.sqrt(252) if excess_returns.std() > 0 else 0
max_dd = result['max_drawdown']

# Sortino
downside = excess_returns[excess_returns < 0]
downside_std = downside.std() * np.sqrt(252) if len(downside) > 0 else ann_vol
sortino = (ann_return - RISK_FREE_RATE) / downside_std if downside_std > 0 else 0
calmar = (ann_return - RISK_FREE_RATE) / abs(max_dd) if max_dd != 0 else 0

# IS-only
is_mask = daily_returns.index <= IS_END
is_rets = daily_returns[is_mask]
is_excess = is_rets - daily_rf
is_sharpe = is_excess.mean() / is_excess.std() * np.sqrt(252) if is_excess.std() > 0 else 0
equity_curve = result['equity_curve'].set_index('date')['portfolio_value']
is_ann_ret = (1 + (equity_curve[is_mask].iloc[-1] / equity_curve[is_mask].iloc[0] - 1)) ** (252 / len(is_rets)) - 1

# OOS-only
oos_mask = daily_returns.index >= OOS_START
oos_rets = daily_returns[oos_mask]
oos_excess = oos_rets - daily_rf
oos_sharpe = oos_excess.mean() / oos_excess.std() * np.sqrt(252) if oos_excess.std() > 0 else 0
oos_ann_ret = (1 + (equity_curve[oos_mask].iloc[-1] / equity_curve[oos_mask].iloc[0] - 1)) ** (252 / len(oos_rets)) - 1

# Comparison: no vol targeting
noTV_rets = result_noTV['daily_returns']
noTV_excess = noTV_rets - daily_rf
noTV_sharpe = noTV_excess.mean() / noTV_excess.std() * np.sqrt(252) if noTV_excess.std() > 0 else 0

print('=' * 75)
print('CORE PERFORMANCE METRICS')
print('=' * 75)
print(f'  Risk-Free Rate:         {RISK_FREE_RATE:.1%}')
print(f'  Target Vol:             {TARGET_VOL:.0%}')
print(f'  Total Return:           {total_return:.2%}')
print(f'  Annualized Return:      {ann_return:.2%}')
print(f'  Annualized Volatility:  {ann_vol:.2%}')
print(f'  Sharpe (raw):           {sharpe_raw:.3f}')
print(f'  Sharpe (RF-adjusted):   {sharpe_adj:.3f}  <-- GATE VALUE')
print(f'  Sortino (RF-adjusted):  {sortino:.3f}')
print(f'  Max Drawdown:           {max_dd:.2%}')
print(f'  Calmar (RF-adjusted):   {calmar:.3f}')
print(f'  Annual Turnover:        {annual_turnover:.0f}% (one-way, weight-diff)')
print(f'  Backtest Days:          {n_days} ({n_days/252:.1f} years)')
print()
print('--- Vol Targeting Impact ---')
print(f'  Sharpe WITHOUT target_vol: {noTV_sharpe:.3f}')
print(f'  Sharpe WITH target_vol:    {sharpe_adj:.3f}')
print(f'  Max DD WITHOUT target_vol: {result_noTV["max_drawdown"]:.2%}')
print(f'  Max DD WITH target_vol:    {max_dd:.2%}')
print()
print('--- IS / OOS Split ---')
print(f'  IS  ({is_rets.index[0].date()} to {IS_END}): Sharpe={is_sharpe:.3f} | Return={is_ann_ret:.2%} | Days={len(is_rets)}')
print(f'  OOS ({OOS_START} to {oos_rets.index[-1].date()}): Sharpe={oos_sharpe:.3f} | Return={oos_ann_ret:.2%} | Days={len(oos_rets)}')

CORE PERFORMANCE METRICS
  Risk-Free Rate:         5.3%
  Target Vol:             12%
  Total Return:           374.86%
  Annualized Return:      12.01%
  Annualized Volatility:  16.24%
  Sharpe (raw):           0.780
  Sharpe (RF-adjusted):   0.454  <-- GATE VALUE
  Sortino (RF-adjusted):  0.503
  Max Drawdown:           -32.00%
  Calmar (RF-adjusted):   0.210
  Annual Turnover:        178% (one-way, weight-diff)
  Backtest Days:          3460 (13.7 years)

--- Vol Targeting Impact ---
  Sharpe WITHOUT target_vol: 0.591
  Sharpe WITH target_vol:    0.454
  Max DD WITHOUT target_vol: -32.00%
  Max DD WITH target_vol:    -32.00%

--- IS / OOS Split ---
  IS  (2011-03-31 to 2021-12-31): Sharpe=0.516 | Return=13.55% | Days=2708
  OOS (2022-01-01 to 2024-12-30): Sharpe=0.180 | Return=6.76% | Days=752


In [8]:
# Cell 10: PSR (Probabilistic Sharpe Ratio)
from backtests.stats.sharpe_tests import probabilistic_sharpe_ratio, sharpe_confidence_interval

psr = probabilistic_sharpe_ratio(daily_returns.values, benchmark_sharpe=0.0, risk_free_rate=RISK_FREE_RATE)
ci_low, ci_point, ci_high = sharpe_confidence_interval(daily_returns.values)

print('=' * 70)
print('PROBABILISTIC SHARPE RATIO (RF-adjusted)')
print('=' * 70)
print(f'  PSR (vs SR=0):       {psr:.4f} ({psr*100:.1f}%)')
print(f'  Gate (> 0.80):       {"PASS" if psr > 0.80 else "FAIL"}')
print(f'  Sharpe 95% CI:       [{ci_low:.3f}, {ci_high:.3f}]')

PROBABILISTIC SHARPE RATIO (RF-adjusted)
  PSR (vs SR=0):       0.9528 (95.3%)
  Gate (> 0.80):       PASS
  Sharpe 95% CI:       [0.316, 1.326]


In [9]:
# Cell 11: Deflated Sharpe Ratio + MinBTL
from backtests.stats.sharpe_tests import deflated_sharpe_ratio
from backtests.stats.minimum_backtest import minimum_backtest_length
from scipy import stats as sp_stats

dsr = deflated_sharpe_ratio(daily_returns.values, n_trials=N_TRIALS, risk_free_rate=RISK_FREE_RATE)

excess = daily_returns.values - RISK_FREE_RATE / 252
obs_sharpe = np.mean(excess) / np.std(excess, ddof=1) * np.sqrt(252)
skew_val = sp_stats.skew(excess)
kurt_val = sp_stats.kurtosis(excess, fisher=True)

min_btl_days = minimum_backtest_length(
    observed_sharpe=obs_sharpe,
    n_trials=N_TRIALS,
    skewness=skew_val,
    kurtosis=kurt_val + 3,
    confidence=0.95,
)
min_btl_years = min_btl_days / 252

print('=' * 70)
print('DEFLATED SHARPE RATIO + MINIMUM BACKTEST LENGTH')
print('=' * 70)
print(f'  Observed Sharpe (RF-adj): {obs_sharpe:.3f}')
print(f'  DSR:                      {dsr:.4f}')
print(f'  Gate (DSR > 0):           {"PASS" if dsr > 0 else "FAIL"}')
print(f'  MinBTL:                   {min_btl_days} days ({min_btl_years:.1f} years)')
print(f'  Actual backtest:          {n_days} days ({n_days/252:.1f} years)')
print(f'  Gate (MinBTL < actual):   {"PASS" if min_btl_days < n_days else "FAIL"}')

DEFLATED SHARPE RATIO + MINIMUM BACKTEST LENGTH
  Observed Sharpe (RF-adj): 0.454
  DSR:                      1.0000
  Gate (DSR > 0):           PASS
  MinBTL:                   6650 days (26.4 years)
  Actual backtest:          3460 days (13.7 years)
  Gate (MinBTL < actual):   FAIL


In [10]:
# Cell 12: Walk-Forward Analysis (rolling-window cov, RF-adj, with vol targeting)
TRAIN_MONTHS = 48
TEST_MONTHS = 12
TRAIN_DAYS = TRAIN_MONTHS * 21
TEST_DAYS = TEST_MONTHS * 21

valid_prices = prices.iloc[WARMUP:]
valid_dates = valid_prices.index
step = TEST_DAYS

wf_results = []
wf_idx = 0

print('Walk-Forward: 48mo train / 12mo test | target_vol=12% | rolling-window cov')
print('=' * 80)

train_start_loc = 0
while True:
    train_end_loc = train_start_loc + TRAIN_DAYS
    test_start_loc = train_end_loc
    test_end_loc = test_start_loc + TEST_DAYS

    if test_end_loc >= len(valid_dates):
        break

    train_start = str(valid_dates[train_start_loc].date())
    train_end = str(valid_dates[train_end_loc].date())
    test_start = str(valid_dates[test_start_loc].date())
    test_end = str(valid_dates[test_end_loc].date())

    wf_config = PortfolioConfig(
        universe=list(prices.columns), signals=[],
        optimization='mean_variance', risk_aversion=RISK_AVERSION,
        max_weight=MAX_WEIGHT, min_weight=MIN_WEIGHT, target_gross=1.0,
        rebalance_frequency=REBALANCE_FREQ, turnover_penalty=0.5,
        initial_cash=100000, commission=0.001,
    )
    wf_builder = PortfolioBuilder(config=wf_config)
    # Rolling-window: only data from training window start
    wf_builder.prices = prices.loc[train_start:]
    wf_builder.signals = {'vol_scaled_momentum': alpha}
    wf_builder.weights = pd.Series(1.0 / n_assets, index=prices.columns)

    wf_result = wf_builder.backtest(
        start_date=test_start,
        end_date=test_end,
        cost_model=cost_model,
        dynamic_reoptimize=True,
        target_vol=TARGET_VOL,
    )

    if wf_result and wf_result.get('n_days', 0) > 20:
        wf_rets = wf_result['daily_returns']
        wf_excess = wf_rets - RISK_FREE_RATE / 252
        wf_sharpe = wf_excess.mean() / wf_excess.std() * np.sqrt(252) if wf_excess.std() > 0 else 0

        wf_results.append({
            'window': wf_idx + 1,
            'train_start': train_start, 'train_end': train_end,
            'test_start': test_start, 'test_end': test_end,
            'oos_sharpe': wf_sharpe,
            'oos_return': wf_result['annualized_return'],
            'oos_max_dd': wf_result['max_drawdown'],
        })
        print(f'  Window {wf_idx+1}: Train {train_start} to {train_end} | '
              f'Test {test_start} to {test_end} | '
              f'SR={wf_sharpe:.3f} | Ret={wf_result["annualized_return"]:.2%} | DD={wf_result["max_drawdown"]:.2%}')

    wf_idx += 1
    train_start_loc += step

wf_df = pd.DataFrame(wf_results)
wf_hit_rate = (wf_df['oos_sharpe'] > 0).mean() if not wf_df.empty else 0
wf_avg_sharpe = wf_df['oos_sharpe'].mean() if not wf_df.empty else 0

print(f'\n{"=" * 70}')
print(f'WALK-FORWARD SUMMARY ({len(wf_df)} windows)')
print(f'{"=" * 70}')
print(f'  Hit rate (OOS SR > 0):  {wf_hit_rate:.1%}')
print(f'  Avg OOS Sharpe:         {wf_avg_sharpe:.3f}')
print(f'  Std OOS Sharpe:         {wf_df["oos_sharpe"].std():.3f}')
print(f'  Gate (hit rate > 55%):  {"PASS" if wf_hit_rate > 0.55 else "FAIL"}')

# Challenge #4: Investigate OOS > IS outlier
# Check if Window 6 (approx COVID recovery 2020-2021) is an outlier
if len(wf_df) > 1:
    max_window = wf_df.loc[wf_df['oos_sharpe'].idxmax()]
    print(f'\n--- Outlier Analysis (Challenge #4) ---')
    print(f'  Best window: #{int(max_window["window"])} ({max_window["test_start"]} to {max_window["test_end"]})')
    print(f'  Best OOS Sharpe: {max_window["oos_sharpe"]:.3f}')

    # Exclude best window
    wf_excl = wf_df[wf_df['window'] != max_window['window']]
    excl_hit = (wf_excl['oos_sharpe'] > 0).mean()
    excl_avg = wf_excl['oos_sharpe'].mean()
    print(f'  Excluding best window: Hit rate={excl_hit:.1%} | Avg SR={excl_avg:.3f}')
    print(f'  Gate without outlier:  {"PASS" if excl_hit > 0.55 else "FAIL"}')

Walk-Forward: 48mo train / 12mo test | target_vol=12% | rolling-window cov


  Window 1: Train 2011-03-30 to 2015-04-02 | Test 2015-04-02 to 2016-04-04 | SR=-0.583 | Ret=-5.62% | DD=-17.33%
  Window 2: Train 2012-03-29 to 2016-04-04 | Test 2016-04-04 to 2017-04-03 | SR=0.431 | Ret=9.86% | DD=-6.23%


  Window 3: Train 2013-04-03 to 2017-04-03 | Test 2017-04-03 to 2018-04-04 | SR=1.342 | Ret=25.89% | DD=-10.01%
  Window 4: Train 2014-04-02 to 2018-04-04 | Test 2018-04-04 to 2019-04-04 | SR=0.236 | Ret=7.93% | DD=-17.01%


  Window 5: Train 2015-04-02 to 2019-04-04 | Test 2019-04-04 to 2020-04-03 | SR=-0.308 | Ret=-9.22% | DD=-30.72%
  Window 6: Train 2016-04-04 to 2020-04-03 | Test 2020-04-03 to 2021-04-06 | SR=2.243 | Ret=57.51% | DD=-8.26%


  Window 7: Train 2017-04-03 to 2021-04-06 | Test 2021-04-06 to 2022-04-04 | SR=1.295 | Ret=24.22% | DD=-6.91%
  Window 8: Train 2018-04-04 to 2022-04-04 | Test 2022-04-04 to 2023-04-05 | SR=-0.184 | Ret=0.88% | DD=-12.85%


  Window 9: Train 2019-04-04 to 2023-04-05 | Test 2023-04-05 to 2024-04-08 | SR=1.504 | Ret=23.70% | DD=-7.12%

WALK-FORWARD SUMMARY (9 windows)
  Hit rate (OOS SR > 0):  66.7%
  Avg OOS Sharpe:         0.664
  Std OOS Sharpe:         0.969
  Gate (hit rate > 55%):  PASS

--- Outlier Analysis (Challenge #4) ---
  Best window: #6 (2020-04-03 to 2021-04-06)
  Best OOS Sharpe: 2.243
  Excluding best window: Hit rate=62.5% | Avg SR=0.467
  Gate without outlier:  PASS


In [11]:
# Cell 13: Cost Sensitivity (1x, 1.5x, 2x, 3x)
cost_multipliers = [0, 1.0, 1.5, 2.0, 3.0]
cost_results = []

for mult in cost_multipliers:
    if mult == 0:
        cm = None
        label = 'No cost'
    else:
        cm = CompositeCostModel(
            models=(
                FixedCostModel(cost_per_trade=FIXED_COST_PER_TRADE * mult),
                ProportionalCostModel(cost_bps=PROP_COST_BPS * mult),
            )
        )
        label = f'{mult}x'

    cs_builder = PortfolioBuilder(config=config)
    cs_builder.prices = prices
    cs_builder.signals = {'vol_scaled_momentum': alpha}
    cs_builder.weights = pd.Series(1.0 / n_assets, index=prices.columns)

    cs_result = cs_builder.backtest(
        start_date=backtest_start, end_date=END_DATE,
        cost_model=cm, dynamic_reoptimize=True,
        target_vol=TARGET_VOL,
    )

    cs_rets = cs_result['daily_returns']
    cs_excess = cs_rets - RISK_FREE_RATE / 252
    cs_sharpe = cs_excess.mean() / cs_excess.std() * np.sqrt(252) if cs_excess.std() > 0 else 0

    cost_results.append({
        'label': label, 'bps': PROP_COST_BPS * mult if mult > 0 else 0,
        'sharpe': cs_sharpe, 'ann_return': cs_result['annualized_return'],
        'max_dd': cs_result['max_drawdown'],
    })

print('COST SENSITIVITY (RF-adjusted, with target_vol=12%)')
print('=' * 70)
print(f'{"Mult":<10} {"Cost(bps)":<10} {"Sharpe":<10} {"Ann Ret":<12} {"Max DD":<10}')
print('-' * 55)
for r in cost_results:
    print(f'{r["label"]:<10} {r["bps"]:<10.0f} {r["sharpe"]:<10.3f} {r["ann_return"]:<12.2%} {r["max_dd"]:<10.2%}')

cs_2x = [r['sharpe'] for r in cost_results if r['label'] == '2.0x'][0]
cs_3x = [r['sharpe'] for r in cost_results if r['label'] == '3.0x'][0]
print(f'\n2x costs: Sharpe={cs_2x:.3f} -> {"PASS" if cs_2x > 0 else "FAIL"}')
print(f'3x costs: Sharpe={cs_3x:.3f} -> {"PASS" if cs_3x > 0 else "FAIL"}')

COST SENSITIVITY (RF-adjusted, with target_vol=12%)
Mult       Cost(bps)  Sharpe     Ann Ret      Max DD    
-------------------------------------------------------
No cost    0          0.588      14.45%       -32.00%   
1.0x       10         0.454      12.01%       -32.00%   
1.5x       15         0.375      10.61%       -32.00%   
2.0x       20         0.297      9.22%        -32.00%   
3.0x       30         0.142      6.48%        -32.00%   

2x costs: Sharpe=0.297 -> PASS
3x costs: Sharpe=0.142 -> PASS


In [12]:
# Cell 14: Regime Analysis
from backtests.walkforward import RegimeAnalyzer

market_df = prices.mean(axis=1).to_frame('close')
ra = RegimeAnalyzer(result, market_df)
regime_metrics = ra.analyze()

print('=' * 75)
print('REGIME ANALYSIS (with target_vol=12%)')
print('=' * 75)
print(f'{"Regime":<20} {"Sharpe":<10} {"Ann Ret":<14} {"Max DD":<10} {"N Days":<8}')
print('-' * 65)

worst_regime_loss = 0
worst_regime_name = 'none'
for name, m in sorted(regime_metrics.items()):
    r_ann = m.get('annualized_return', 0)
    r_vol = m.get('volatility', 1)
    r_sharpe = (r_ann - RISK_FREE_RATE) / r_vol if r_vol > 0 else 0
    r_dd = m.get('max_drawdown', 0)
    r_days = m.get('n_days', 0)
    print(f'{name:<20} {r_sharpe:<10.3f} {r_ann:<14.2%} {r_dd:<10.2%} {r_days:<8}')
    if r_ann < worst_regime_loss:
        worst_regime_loss = r_ann
        worst_regime_name = name

# Compare with no vol targeting
ra_noTV = RegimeAnalyzer(result_noTV, market_df)
regime_noTV = ra_noTV.analyze()

print(f'\nWorst regime: {worst_regime_name} ({worst_regime_loss:.2%} ann)')
print(f'Gate (> -15%): {"PASS" if worst_regime_loss > -0.15 else "FAIL"}')

# Show vol targeting impact on worst regime
if 'trend_bear' in regime_noTV:
    noTV_bear = regime_noTV['trend_bear'].get('annualized_return', 0)
    print(f'\n--- Vol Targeting Impact on Bear Regime ---')
    print(f'  WITHOUT target_vol: {noTV_bear:.2%} ann')
    print(f'  WITH target_vol:    {worst_regime_loss:.2%} ann')
    print(f'  Improvement:        {worst_regime_loss - noTV_bear:+.2%}')

REGIME ANALYSIS (with target_vol=12%)
Regime               Sharpe     Ann Ret        Max DD     N Days  
-----------------------------------------------------------------
trend_bear           -2.024     -48.12%        -87.46%    798     
trend_bull           3.209      41.24%         -9.83%     2661    
vol_high_vol         0.245      10.40%         -32.00%    1645    
vol_low_vol          0.800      13.67%         -8.99%     1814    

Worst regime: trend_bear (-48.12% ann)
Gate (> -15%): FAIL

--- Vol Targeting Impact on Bear Regime ---
  WITHOUT target_vol: -52.94% ann
  WITH target_vol:    -48.12% ann
  Improvement:        +4.82%


In [13]:
# Cell 15: Parameter Sensitivity (+/-20%, +/-40%)
def run_sensitivity(vol_lb, mom_lb, risk_av):
    from backtests.strategies.signals import VolatilitySignal, MomentumSignal, MeanReversionSignal
    vs = VolatilitySignal(lookback=vol_lb)
    rv = vs.compute(prices)
    ms = MomentumSignal(lookback=mom_lb, skip=21)
    rm = ms.compute(prices)
    tv = prices.pct_change().rolling(mom_lb, min_periods=mom_lb).std() * np.sqrt(252)
    rsm = rm / tv.replace(0, np.nan)
    rsm = rsm.replace([np.inf, -np.inf], np.nan)
    mrs = MeanReversionSignal(lookback=20)
    rmr = mrs.compute(prices)
    zv = expanding_zscore(rv)
    zm = expanding_zscore(rsm)
    zmr = expanding_zscore(rmr)
    a = WEIGHT_INVVOL * zv + WEIGHT_MOMENTUM * zm + WEIGHT_MEANREV * zmr
    warmup = max(vol_lb, mom_lb) + MIN_PERIODS_ZSCORE
    a = a.iloc[warmup:]

    sc = PortfolioConfig(
        universe=list(prices.columns), signals=[],
        optimization='mean_variance', risk_aversion=risk_av,
        max_weight=MAX_WEIGHT, min_weight=MIN_WEIGHT, target_gross=1.0,
        rebalance_frequency=REBALANCE_FREQ, turnover_penalty=0.5,
        initial_cash=100000, commission=0.001,
    )
    sb = PortfolioBuilder(config=sc)
    sb.prices = prices
    sb.signals = {'vol_scaled_momentum': a}
    sb.weights = pd.Series(1.0 / n_assets, index=prices.columns)
    sr = sb.backtest(
        start_date=str(a.index[0].date()), end_date=END_DATE,
        cost_model=cost_model, dynamic_reoptimize=True,
        target_vol=TARGET_VOL,
    )
    sr_excess = sr['daily_returns'] - RISK_FREE_RATE / 252
    sr_sharpe = sr_excess.mean() / sr_excess.std() * np.sqrt(252) if sr_excess.std() > 0 else 0
    return sr_sharpe, sr['annualized_return'], sr['max_drawdown']

param_tests = [
    ('Vol LB -40%', 36, 252, 3.0), ('Vol LB -20%', 48, 252, 3.0),
    ('Base', 60, 252, 3.0),
    ('Vol LB +20%', 72, 252, 3.0), ('Vol LB +40%', 84, 252, 3.0),
    ('Mom LB -40%', 60, 151, 3.0), ('Mom LB -20%', 60, 202, 3.0),
    ('Mom LB +20%', 60, 302, 3.0), ('Mom LB +40%', 60, 353, 3.0),
    ('RiskAv -40%', 60, 252, 1.8), ('RiskAv -20%', 60, 252, 2.4),
    ('RiskAv +20%', 60, 252, 3.6), ('RiskAv +40%', 60, 252, 4.2),
]

print('PARAMETER SENSITIVITY (RF-adj, target_vol=12%)')
print('=' * 75)
print(f'{"Variation":<18} {"Vol":<6} {"Mom":<6} {"RA":<6} {"Sharpe":<10} {"Return":<12} {"MaxDD":<10}')
print('-' * 70)
for label, vlb, mlb, ra in param_tests:
    sh, ret, dd = run_sensitivity(vlb, mlb, ra)
    print(f'{label:<18} {vlb:<6} {mlb:<6} {ra:<6.1f} {sh:<10.3f} {ret:<12.2%} {dd:<10.2%}')

PARAMETER SENSITIVITY (RF-adj, target_vol=12%)
Variation          Vol    Mom    RA     Sharpe     Return       MaxDD     
----------------------------------------------------------------------


Vol LB -40%        36     252    3.0    0.453      11.85%       -27.49%   


Vol LB -20%        48     252    3.0    0.425      11.42%       -27.59%   


Base               60     252    3.0    0.454      12.01%       -32.00%   


Vol LB +20%        72     252    3.0    0.421      11.46%       -31.76%   


...

Mom LB +40%        60     353    3.0    0.588      14.53%       -32.01%   


RiskAv -40%        60     252    1.8    0.454      12.01%       -32.00%   


RiskAv -20%        60     252    2.4    0.454      12.02%       -32.00%   


RiskAv +20%        60     252    3.6    0.453      12.00%       -32.00%   


RiskAv +40%        60     252    4.2    0.452      11.98%       -32.00%   


In [14]:
# Cell 16: Weight Distribution & Binding Constraints (Challenge #5)

print('WEIGHT DISTRIBUTION ANALYSIS')
print('=' * 70)

# Check how often max_weight is binding
at_max = (weight_matrix >= MAX_WEIGHT - 0.001).sum(axis=0)
total_days = len(weight_matrix)

print(f'Max weight constraint: {MAX_WEIGHT:.2f}')
print(f'\nDays at max weight by ticker:')
binding = at_max[at_max > 0].sort_values(ascending=False)
for ticker, days in binding.items():
    print(f'  {ticker}: {days} days ({days/total_days:.1%})')

# Average number of stocks at max weight per day
avg_at_max = (weight_matrix >= MAX_WEIGHT - 0.001).sum(axis=1).mean()
print(f'\nAvg stocks at max weight per day: {avg_at_max:.1f}')
print(f'This means the optimizer wants to concentrate but is constrained.')

# Show weight distribution at a few dates
sample_dates = weight_matrix.index[::len(weight_matrix)//4]
print(f'\nWeight distribution at sample dates:')
for d in sample_dates:
    w = weight_matrix.loc[d]
    nonzero = w[w > 0.001]
    print(f'  {d.date()}: {len(nonzero)} stocks | max={w.max():.3f} | min(nonzero)={nonzero.min():.3f} | gross={w.sum():.3f}')

# Why risk_aversion has no effect: if max_weight is always binding,
# the optimizer is constrained to max_weight for top stocks regardless of RA
print(f'\nConclusion: The max_weight={MAX_WEIGHT} constraint IS binding.')
print(f'Risk aversion has minimal effect because the optimizer cannot')
print(f'concentrate beyond {MAX_WEIGHT:.0%} per stock regardless of RA value.')
print(f'The portfolio is effectively constrained-optimal, not mean-variance optimal.')

WEIGHT DISTRIBUTION ANALYSIS
Max weight constraint: 0.12

Days at max weight by ticker:
  JPM: 1215 days (35.1%)
  BAC: 1004 days (29.0%)
  MA: 962 days (27.8%)
  AMZN: 844 days (24.4%)
  ACN: 841 days (24.3%)
  V: 838 days (24.2%)
  AVGO: 802 days (23.2%)
  MSFT: 752 days (21.7%)
  CSCO: 717 days (20.7%)
  BRK-B: 588 days (17.0%)
  GOOGL: 504 days (14.6%)
  TMO: 504 days (14.6%)
  AAPL: 422 days (12.2%)
  HD: 381 days (11.0%)
  TXN: 376 days (10.9%)
  WMT: 340 days (9.8%)
  MDT: 338 days (9.8%)
  UNH: 336 days (9.7%)
  MRK: 296 days (8.6%)
  LIN: 255 days (7.4%)
  JNJ: 254 days (7.3%)
  PEP: 250 days (7.2%)
  NEE: 250 days (7.2%)
  CVX: 211 days (6.1%)
  COST: 211 days (6.1%)
  PG: 209 days (6.0%)
  KO: 42 days (1.2%)
  XOM: 42 days (1.2%)

Avg stocks at max weight per day: 4.0
This means the optimizer wants to concentrate but is constrained.

Weight distribution at sample dates:
  2011-03-31: 9 stocks | max=0.120 | min(nonzero)=0.040 | gross=1.000
  2014-09-09: 9 stocks | max=0.120 |

In [15]:
# Cell 17: Decay & Capacity Analysis
from backtests.stats.decay_analysis import rolling_sharpe, strategy_half_life

rs_3m = rolling_sharpe(daily_returns, window=63)
rs_12m = rolling_sharpe(daily_returns, window=252)
half_life = strategy_half_life(daily_returns, window=252)

print('=' * 70)
print('DECAY & CAPACITY ANALYSIS')
print('=' * 70)
print(f'  Rolling Sharpe (3M) mean:    {rs_3m.mean():.3f}')
print(f'  Rolling Sharpe (12M) mean:   {rs_12m.mean():.3f}')
if half_life:
    print(f'  Strategy Half-Life:          {half_life:.1f} yrs')
    print(f'  Gate (> 2yr):                {"PASS" if half_life > 2 else "FAIL"}')
else:
    print(f'  Strategy Half-Life:          No decay detected')
    print(f'  Gate (> 2yr):                PASS (no decay)')

DECAY & CAPACITY ANALYSIS
  Rolling Sharpe (3M) mean:    1.314
  Rolling Sharpe (12M) mean:   1.038
  Strategy Half-Life:          No decay detected
  Gate (> 2yr):                PASS (no decay)


In [16]:
# Cell 18: Equal-Weight Benchmark Comparison (Challenge #6)

# Equal-weight buy-and-hold: simple average of daily returns across all tickers
ew_returns = prices.pct_change().mean(axis=1)
ew_returns = ew_returns.reindex(daily_returns.index).dropna()

# Align on common dates
common_idx = daily_returns.index.intersection(ew_returns.index)
strat_r = daily_returns.loc[common_idx]
ew_r = ew_returns.loc[common_idx]

# Also download SPY
spy_raw = yf.download('SPY', start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
if isinstance(spy_raw.columns, pd.MultiIndex):
    spy_prices = spy_raw['Close'].squeeze()
else:
    spy_prices = spy_raw['Close']
spy_returns = spy_prices.pct_change().dropna()
spy_r = spy_returns.reindex(common_idx).dropna()
common_idx2 = common_idx.intersection(spy_r.index)

print('=' * 75)
print('BENCHMARK COMPARISON')
print('=' * 75)

for name, bench_r in [('Equal-Weight (same 28 stocks)', ew_r), ('SPY', spy_r.reindex(common_idx2))]:
    ci = strat_r.index.intersection(bench_r.dropna().index)
    sr = strat_r.loc[ci]
    br = bench_r.loc[ci]

    sr_excess = sr - RISK_FREE_RATE / 252
    br_excess = br - RISK_FREE_RATE / 252
    active = sr - br

    s_sharpe = sr_excess.mean() / sr_excess.std() * np.sqrt(252)
    b_sharpe = br_excess.mean() / br_excess.std() * np.sqrt(252)
    te = active.std() * np.sqrt(252)
    ir = active.mean() * 252 / te if te > 0 else 0
    cov_m = np.cov(sr, br)
    beta = cov_m[0, 1] / cov_m[1, 1] if cov_m[1, 1] > 0 else 0
    alpha_ann = (sr.mean() - beta * br.mean()) * 252

    # Cumulative returns
    strat_cum = (1 + sr).cumprod().iloc[-1] - 1
    bench_cum = (1 + br).cumprod().iloc[-1] - 1

    print(f'\nvs {name}:')
    print(f'  Strategy: SR={s_sharpe:.3f} | Cum={strat_cum:.2%}')
    print(f'  Bench:    SR={b_sharpe:.3f} | Cum={bench_cum:.2%}')
    print(f'  Alpha:    {alpha_ann:.2%} (annualized)')
    print(f'  Beta:     {beta:.3f}')
    print(f'  IR:       {ir:.3f}')
    print(f'  TE:       {te:.2%}')

BENCHMARK COMPARISON

vs Equal-Weight (same 28 stocks):
  Strategy: SR=0.454 | Cum=374.86%
  Bench:    SR=0.801 | Cum=940.36%
  Alpha:    -3.32% (annualized)
  Beta:     0.869
  IR:       -0.703
  TE:       8.16%

vs SPY:
  Strategy: SR=0.454 | Cum=374.86%
  Bench:    SR=0.518 | Cum=469.94%
  Alpha:    1.21% (annualized)
  Beta:     0.811
  IR:       -0.161
  TE:       9.10%


## Summary -- Quantitative Gates (Round 2)

All Sharpe ratios are RF-adjusted (rf=5.3%). Turnover = one-way weight diffs. Vol targeting at 12%.

In [17]:
# Cell 19: Summary Gate Table

gate_sharpe_is = is_sharpe
gate_dsr = dsr
gate_psr = psr
gate_wf_hit = wf_hit_rate
gate_2x = cs_2x
gate_3x = cs_3x
gate_worst = worst_regime_loss
gate_hl = half_life if half_life else float('inf')
gate_minbtl = min_btl_days
gate_maxdd = result['max_drawdown']
gate_turnover = annual_turnover

gates = [
    ('Sharpe IS (RF-adj)', f'{gate_sharpe_is:.3f}', '> 0.5', gate_sharpe_is > 0.5),
    ('Deflated Sharpe', f'{gate_dsr:.4f}', '> 0', gate_dsr > 0),
    ('PSR', f'{gate_psr*100:.1f}%', '> 80%', gate_psr > 0.80),
    ('WF Hit Rate', f'{gate_wf_hit*100:.1f}%', '> 55%', gate_wf_hit > 0.55),
    ('Survives 2x Costs', f'{gate_2x:.3f}', 'SR > 0', gate_2x > 0),
    ('3x Cost Sensitivity', f'{gate_3x:.3f}', 'SR > 0', gate_3x > 0),
    ('Worst Regime Loss', f'{gate_worst:.2%}', '> -15%', gate_worst > -0.15),
    ('Strategy Half-Life', f'{gate_hl:.1f} yrs' if gate_hl != float('inf') else 'No decay', '> 2 yrs', gate_hl > 2),
    ('MinBTL', f'{gate_minbtl/252:.1f} yrs', f'< {n_days/252:.1f} yrs', gate_minbtl < n_days),
    ('Max Drawdown', f'{gate_maxdd:.2%}', '< 25%', abs(gate_maxdd) < 0.25),
    ('Annual Turnover', f'{gate_turnover:.0f}%', '< 150%', gate_turnover < 150),
]

n_pass = sum(1 for g in gates if g[3])

print('=' * 75)
print('QUANTITATIVE GATES SUMMARY (R2 -- RF-adj, target_vol=12%)')
print('=' * 75)
print(f'{"Gate":<25} {"Value":<18} {"Threshold":<14} {"Status":<8}')
print('-' * 75)
for name, val, thresh, passed in gates:
    print(f'{name:<25} {val:<18} {thresh:<14} {"[PASS]" if passed else "[FAIL]"}')

print(f'\n{"=" * 75}')
print(f'GATES PASSED: {n_pass} / {len(gates)}')
print(f'{"=" * 75}')

# Additional evidence
print(f'\n--- Key Round 2 Evidence ---')
print(f'  Vol targeting impact on max DD:')
print(f'    Without: {result_noTV["max_drawdown"]:.2%}')
print(f'    With:    {result["max_drawdown"]:.2%}')
print(f'  IS Sharpe: {is_sharpe:.3f} | OOS Sharpe: {oos_sharpe:.3f}')
print(f'  WF Avg OOS Sharpe: {wf_avg_sharpe:.3f}')
print(f'  Max weight binding: YES ({avg_at_max:.0f} stocks at {MAX_WEIGHT:.0%} per day)')
print(f'  MTUM-like overlap: strategy is constrained-optimal, not pure momentum')

print(f'\n--- Researcher Self-Assessment ---')
print(f'The vol-targeting mechanism (Moreira & Muir 2017) reduces gross exposure')
print(f'during high-vol periods. The key question is whether this sufficiently')
print(f'reduces drawdowns and regime losses to pass the remaining gates.')
print(f'Turnover may increase due to vol-scaling adjustments on top of')
print(f'dynamic reoptimization -- this is a tradeoff between risk control')
print(f'and transaction costs.')

QUANTITATIVE GATES SUMMARY (R2 -- RF-adj, target_vol=12%)
Gate                      Value              Threshold      Status  
---------------------------------------------------------------------------
Sharpe IS (RF-adj)        0.516              > 0.5          [PASS]
Deflated Sharpe           1.0000             > 0            [PASS]
PSR                       95.3%              > 80%          [PASS]
WF Hit Rate               66.7%              > 55%          [PASS]
Survives 2x Costs         0.297              SR > 0         [PASS]
3x Cost Sensitivity       0.142              SR > 0         [PASS]
Worst Regime Loss         -48.12%            > -15%         [FAIL]
Strategy Half-Life        No decay           > 2 yrs        [PASS]
MinBTL                    26.4 yrs           < 13.7 yrs     [FAIL]
Max Drawdown              -32.00%            < 25%          [FAIL]
Annual Turnover           178%               < 150%         [FAIL]

GATES PASSED: 7 / 11

--- Key Round 2 Evidence ---
  Vol ta